# Обучение LoRA для NeuroMango (Unsloth Llama-3)
1. Загрузите этот блокнот в Google Colab.
2. Включите GPU (Среда выполнения -> Сменить среду выполнения -> T4 GPU).
3. Нажмите на значок папки слева и загрузите туда ваш `dataset.jsonl` (сгенерированный скриптом `generate_dataset.py`).
4. Выполняйте ячейки по очереди!

In [ ]:
# 1. Установка Unsloth (быстрая библиотека для LoRA)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
# 2. Загрузка базовой модели
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

In [ ]:
# 3. Подключение LoRA адаптеров
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

In [ ]:
# 4. Загрузка вашего датасета
alpaca_prompt = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    outputs      = examples["output"]
    texts = []
    for instruction, output in zip(instructions, outputs):
        text = alpaca_prompt.format(instruction, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts }

from datasets import load_dataset
dataset = load_dataset("json", data_files="dataset.jsonl", split = "train")
dataset = dataset.map(formatting_prompts_func, batched = True)

In [ ]:
# 5. Обучение!
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # Увеличьте до 200-500 для реального обучения!
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)
trainer_stats = trainer.train()

In [ ]:
# 6. Сохранение в формате GGUF для LM Studio
model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")
print("\n✅ ГОТОВО! Слева в файлах Colab откройте папку model/ и скачайте файл .gguf")